In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
%python
source_file=f"{landing_folder_path}/races.csv"
table_name=f"{catalog_name}.{bronze_schema}.races"

In [0]:
%python
from pyspark.sql.types import StructType, StructField, StringType,IntegerType,DateType

# Define schema for races file with StructType and StructField
races_schema = StructType([
    StructField("season", IntegerType(), True),
    StructField("round", IntegerType(), True),
    StructField("url", StringType(), True),
    StructField("raceName", StringType(), True),
    StructField("date", DateType(), True),
    StructField("circuitId", StringType(), True)
])

# Read races file from volume with defined schema
races_df = (spark.read
    .schema(races_schema)
    .option("header", True)
    .option("mode", "FAILFAST")
    .csv(source_file)
)

#display(races_df)

In [0]:
%python
# Add ingestion timestamp and source file path metadata
races_with_metadata_df = add_ingestion_metadata(races_df)

display(races_with_metadata_df)

In [0]:
%python
# Write races data to bronze layer table
(races_with_metadata_df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(table_name)
)

print(f"✅ Races data successfully written to {table_name} table")

In [0]:
SELECT 
  season,
  round,
  raceName,
  date,
  circuitId,
  ingestion_timestamp,
  source_file_path
FROM formula1.bronze.races
ORDER BY season DESC, round DESC
LIMIT 10